# M5 — Hybrid Recommender + Evaluation

Owner: Member 5 (Collaborative Filtering + Hybrid + Evaluation).
Inputs: M2 PostgreSQL DWH, M3 ranked rules CSV, M4 cluster assignments CSV.
Outputs (under `outputs/m5/`):
1. `comparison_table.csv` — 6 systems × {P@10, R@10, HR@10, Coverage, runtime}
2. `per_user_metrics.csv` — long-format per-user metrics for significance tests
3. `significance_results.csv` — Wilcoxon + bootstrap CI + Cliff's delta for Hybrid vs each other system
4. `recommendations_sample.csv` — 20 sampled users × 6 systems × top-10 (for paper/slides)

**Prerequisites:** local Postgres with `olist_dwh` restored from `olist_dwh.dump`; `psycopg2-binary`, `scikit-learn`, `scipy` installed.

## Section 1 — Setup & data loading

In [ ]:
import os, sys, time, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sqlalchemy import text, create_engine

DB_URL = os.environ.get('DATABASE_URL', 'postgresql://localhost/olist_dwh')
engine = create_engine(DB_URL, future=True)

interactions = pd.read_sql(text(
    'SELECT customer_id, product_id, orderdate FROM fact_orderitems'
), engine)
customers = pd.read_sql(text(
    'SELECT customer_id, customer_unique_id FROM dim_customers'
), engine)
products = pd.read_sql(text(
    'SELECT product_id, product_category_name_english, category_group_id FROM dim_products'
), engine)
prices = pd.read_sql(text(
    'SELECT product_id, AVG(price) AS avg_price FROM fact_orderitems GROUP BY product_id'
), engine)

RULES_PATH    = '../outputs/rules/ranked_rules_for_m5_with_segments.csv'
CLUSTERS_PATH = '../outputs/clustering/customer_cluster_assignments.csv'
rules    = pd.read_csv(RULES_PATH)
clusters = pd.read_csv(CLUSTERS_PATH)

interactions = interactions.merge(customers, on='customer_id', how='left')
interactions['orderdate'] = pd.to_datetime(interactions['orderdate'])

print(f'interactions: {len(interactions):,} rows  '
      f'unique customers: {interactions["customer_unique_id"].nunique():,}  '
      f'products: {interactions["product_id"].nunique():,}')
print(f'rules: {len(rules)} rows  clusters: {len(clusters):,} rows')

## Section 2 — Chronological train/test split
Cutoff: 2018-04-01 (M2 handoff suggestion). Test users = customer_unique_ids present in both halves.

In [ ]:
CUTOFF = pd.Timestamp('2018-04-01')

train = interactions[interactions['orderdate'] < CUTOFF].copy()
test  = interactions[interactions['orderdate'] >= CUTOFF].copy()

train_users    = set(train['customer_unique_id'].dropna().unique())
test_users_all = set(test['customer_unique_id'].dropna().unique())
test_users     = sorted(train_users & test_users_all)

train_user_items = train.groupby('customer_unique_id')['product_id'].apply(set).to_dict()
test_user_items  = test.groupby('customer_unique_id')['product_id'].apply(set).to_dict()

print(f'train: {len(train):,} rows, {len(train_users):,} unique customers')
print(f'test : {len(test):,} rows, {len(test_users_all):,} unique customers')
print(f'test users (in both train and test): {len(test_users):,}')
if len(test_users) == 0:
    raise RuntimeError('No test users with both train and test interactions — check cutoff.')

## Section 3 — Evaluation harness (P@K, R@K, Hit Rate, Coverage)

In [ ]:
def precision_at_k(predicted, actual, k=10):
    if not predicted: return 0.0
    return len(set(predicted[:k]) & actual) / k

def recall_at_k(predicted, actual, k=10):
    if not actual: return 0.0
    return len(set(predicted[:k]) & actual) / len(actual)

def hit_at_k(predicted, actual, k=10):
    return 1 if set(predicted[:k]) & actual else 0

def coverage(all_predictions, catalog_size):
    union = set()
    for preds in all_predictions:
        union.update(preds)
    return len(union) / catalog_size if catalog_size else 0.0

def run_eval(recommender_fn, name, test_users, test_user_items, catalog_size, k=10):
    t0 = time.time()
    rows, all_preds = [], []
    for u in test_users:
        preds = recommender_fn(u, k)
        all_preds.append(preds)
        actual = test_user_items.get(u, set())
        rows.append({
            'customer_unique_id': u,
            'precision_at_k': precision_at_k(preds, actual, k),
            'recall_at_k':    recall_at_k(preds, actual, k),
            'hit':            hit_at_k(preds, actual, k),
        })
    elapsed = time.time() - t0
    per_user = pd.DataFrame(rows)
    aggregate = {
        'system': name,
        'precision_at_10': per_user['precision_at_k'].mean(),
        'recall_at_10':    per_user['recall_at_k'].mean(),
        'hit_rate_at_10':  per_user['hit'].mean(),
        'coverage':        coverage(all_preds, catalog_size),
        'runtime_seconds': round(elapsed, 2),
        'n_users':         len(per_user),
    }
    print(f'{name:22s} P@10={aggregate["precision_at_10"]:.4f}  '
          f'R@10={aggregate["recall_at_10"]:.4f}  '
          f'HR={aggregate["hit_rate_at_10"]:.4f}  '
          f'Cov={aggregate["coverage"]:.4f}  '
          f't={elapsed:.1f}s  n={len(per_user)}')
    return aggregate, per_user

## Section 4 — Non-ML baselines (most-popular, category-popular)

In [ ]:
popular_items = train['product_id'].value_counts().index.tolist()

def most_popular_recommend(user, k=10):
    bought = train_user_items.get(user, set())
    return [p for p in popular_items if p not in bought][:k]

prod_to_cat = products.set_index('product_id')['category_group_id'].to_dict()
train_with_cat = train.copy()
train_with_cat['category_group_id'] = train_with_cat['product_id'].map(prod_to_cat)

cat_to_popular = (
    train_with_cat.dropna(subset=['category_group_id'])
                  .groupby('category_group_id')['product_id']
                  .apply(lambda s: s.value_counts().index.tolist())
                  .to_dict()
)

user_top_cat = (
    train_with_cat.dropna(subset=['category_group_id'])
                  .groupby(['customer_unique_id', 'category_group_id'])
                  .size().reset_index(name='n')
                  .sort_values(['customer_unique_id', 'n'], ascending=[True, False])
                  .drop_duplicates('customer_unique_id')
                  .set_index('customer_unique_id')['category_group_id']
                  .to_dict()
)

def category_popular_recommend(user, k=10):
    bought = train_user_items.get(user, set())
    cat = user_top_cat.get(user)
    if cat is None or cat not in cat_to_popular:
        return most_popular_recommend(user, k)
    return [p for p in cat_to_popular[cat] if p not in bought][:k]

catalog_size = train['product_id'].nunique()
print(f'catalog size (train): {catalog_size:,}')
print(f'global top-5 popular products: {popular_items[:5]}')

## Section 5 — Item-based collaborative filtering
Sparse user×item matrix → top-N item-item cosine similarity (truncated to 200 neighbours per item to fit in memory).

In [ ]:
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity

train_unique = train[['customer_unique_id', 'product_id']].dropna().drop_duplicates()
u2i = {u: i for i, u in enumerate(train_unique['customer_unique_id'].unique())}
p2i = {p: i for i, p in enumerate(train_unique['product_id'].unique())}
i2p = {i: p for p, i in p2i.items()}

rows = train_unique['customer_unique_id'].map(u2i).to_numpy()
cols = train_unique['product_id'].map(p2i).to_numpy()
data = np.ones(len(rows), dtype=np.float32)
ui_matrix = csr_matrix((data, (rows, cols)), shape=(len(u2i), len(p2i)))
iu_matrix = ui_matrix.T.tocsr()
print(f'user-item: {ui_matrix.shape}  nnz={ui_matrix.nnz:,}')

N_NEIGHBORS = 200

def topk_neighbors_sparse(item_matrix, n_neighbors=200, chunk=500):
    n = item_matrix.shape[0]
    neighbors = {}
    for start in range(0, n, chunk):
        end = min(start + chunk, n)
        block = cosine_similarity(item_matrix[start:end], item_matrix, dense_output=True)
        for r in range(end - start):
            block[r, start + r] = 0.0
        for r in range(end - start):
            row = block[r]
            if row.max() == 0:
                neighbors[start + r] = []
                continue
            k_eff = min(n_neighbors, n - 1)
            top = np.argpartition(-row, k_eff)[:k_eff]
            top = top[np.argsort(-row[top])]
            top = top[row[top] > 0]
            neighbors[start + r] = [(int(j), float(row[j])) for j in top]
    return neighbors

print('Computing item-item similarity (chunked)...')
_t0 = time.time()
item_neighbors = topk_neighbors_sparse(iu_matrix, n_neighbors=N_NEIGHBORS, chunk=500)
print(f'  done in {time.time()-_t0:.1f}s; edges={sum(len(v) for v in item_neighbors.values()):,}')

def cf_recommend(user, k=10):
    bought = train_user_items.get(user, set())
    bought_idx = [p2i[p] for p in bought if p in p2i]
    if not bought_idx:
        return most_popular_recommend(user, k)
    bought_set = set(bought_idx)
    scores = {}
    for src in bought_idx:
        for (nbr, sim) in item_neighbors.get(src, []):
            if nbr in bought_set:
                continue
            scores[nbr] = scores.get(nbr, 0.0) + sim
    if not scores:
        return most_popular_recommend(user, k)
    top = sorted(scores.items(), key=lambda x: -x[1])[:k]
    return [i2p[i] for i, _ in top]

## Section 6 — Content-based fallback
Per-product features: one-hot(category_group_id) + one-hot(price_bucket from qcut on avg price). Item-item cosine similarity, top-200 neighbours per item.

In [ ]:
prod_prices = prices.copy()
prod_prices['price_bucket'] = pd.qcut(
    prod_prices['avg_price'].rank(method='first'), q=5, labels=False, duplicates='drop'
)

catalog_items = train['product_id'].unique()
content_feat = (products[['product_id', 'category_group_id']]
                .merge(prod_prices[['product_id', 'price_bucket']], on='product_id', how='left'))
content_feat = content_feat[content_feat['product_id'].isin(catalog_items)].reset_index(drop=True)
content_feat['category_group_id'] = content_feat['category_group_id'].fillna(-1).astype(int)
content_feat['price_bucket']      = content_feat['price_bucket'].fillna(-1).astype(int)

cat_d   = pd.get_dummies(content_feat['category_group_id'], prefix='cat')
price_d = pd.get_dummies(content_feat['price_bucket'],      prefix='price')
feat_matrix = pd.concat([cat_d, price_d], axis=1).to_numpy(dtype=np.float32)
feat_sparse = csr_matrix(feat_matrix)

content_p2i = {p: i for i, p in enumerate(content_feat['product_id'].values)}
content_i2p = {i: p for p, i in content_p2i.items()}
print(f'content feature matrix: {feat_matrix.shape}')

print('Computing content item-item similarity (chunked)...')
_t0 = time.time()
content_neighbors = topk_neighbors_sparse(feat_sparse, n_neighbors=200, chunk=500)
print(f'  done in {time.time()-_t0:.1f}s')

def content_recommend(user, k=10):
    bought = train_user_items.get(user, set())
    bought_idx = [content_p2i[p] for p in bought if p in content_p2i]
    if not bought_idx:
        return most_popular_recommend(user, k)
    bought_set = set(bought_idx)
    scores = {}
    for src in bought_idx:
        for (nbr, sim) in content_neighbors.get(src, []):
            if nbr in bought_set:
                continue
            scores[nbr] = scores.get(nbr, 0.0) + sim
    if not scores:
        return most_popular_recommend(user, k)
    top = sorted(scores.items(), key=lambda x: -x[1])[:k]
    return [content_i2p[i] for i, _ in top]

## Section 7 — Rules-based systems (Apriori-only, FP-Growth-only)
Each uses the M3 rules CSV filtered to its own algorithm, on the user's **most-recent train purchase** as the query item. Topped up with most-popular when the rule index has no match (otherwise coverage is ~0%).

In [ ]:
apriori_idx = (
    rules[(rules['algorithm'] == 'Apriori') & (rules['condition'] == 'all')]
        .sort_values('rank')
        .groupby('query_item')['recommended_item'].apply(list).to_dict()
)
fpgrowth_idx = (
    rules[(rules['algorithm'] == 'FP-Growth') & (rules['condition'] == 'all')]
        .sort_values('rank')
        .groupby('query_item')['recommended_item'].apply(list).to_dict()
)

user_last_product = (
    train.sort_values('orderdate')
         .drop_duplicates('customer_unique_id', keep='last')
         .set_index('customer_unique_id')['product_id'].to_dict()
)

def rules_recommend(user, rule_index, k=10):
    bought = train_user_items.get(user, set())
    last_p = user_last_product.get(user)
    recs = []
    if last_p is not None and last_p in rule_index:
        recs = [p for p in rule_index[last_p] if p not in bought][:k]
    if len(recs) < k:
        fillers = [p for p in popular_items if p not in bought and p not in recs]
        recs += fillers[:k - len(recs)]
    return recs[:k]

def apriori_recommend(user, k=10):
    return rules_recommend(user, apriori_idx, k)

def fpgrowth_recommend(user, k=10):
    return rules_recommend(user, fpgrowth_idx, k)

print(f'apriori_idx query_items:  {len(apriori_idx)}')
print(f'fpgrowth_idx query_items: {len(fpgrowth_idx)}')

## Section 8 — Hybrid via Reciprocal Rank Fusion (k=60)
Components: (a) holiday/seasonal rules — *empty, skipped*; (b) per-segment rules; (c) item-CF; (d) content fallback.
Rules CSV is **deduplicated** first to avoid the Apriori/FP-Growth/ECLAT triplication.

In [ ]:
RRF_K = 60  # Cormack et al. 2009 default

rules_dedup = (
    rules.assign(_priority=rules['algorithm'].map({'FP-Growth': 0, 'Apriori': 1, 'ECLAT-style': 2}).fillna(3))
         .sort_values(['query_item', 'recommended_item', 'condition', 'segment_id', '_priority'])
         .drop_duplicates(subset=['query_item', 'recommended_item', 'condition', 'segment_id'], keep='first')
         .drop(columns=['_priority'])
)
print(f'rules before dedup: {len(rules)}  after: {len(rules_dedup)}  query_items: {rules_dedup["query_item"].nunique()}')

segment_rules_idx = {}
seg_rows = rules_dedup[rules_dedup['condition'] == 'segment'].copy()
if len(seg_rows):
    seg_rows['segment_id'] = seg_rows['segment_id'].astype(int)
    for (seg, q), grp in seg_rows.sort_values('rank').groupby(['segment_id', 'query_item']):
        segment_rules_idx[(int(seg), q)] = grp['recommended_item'].tolist()
print(f'segment rules entries: {len(segment_rules_idx)}')

last_order_cid = (
    interactions.sort_values('orderdate')
                .drop_duplicates('customer_unique_id', keep='last')
                .set_index('customer_unique_id')['customer_id'].to_dict()
)
cid_to_cluster = clusters.set_index('customer_id')['cluster_id'].to_dict()

def user_cluster(user):
    cid = last_order_cid.get(user)
    if cid is None:
        return None
    c = cid_to_cluster.get(cid)
    return None if c is None else int(c)

def segment_rules_component(user, k=10):
    seg = user_cluster(user)
    last_p = user_last_product.get(user)
    if seg is None or last_p is None:
        return []
    recs = segment_rules_idx.get((seg, last_p), [])
    bought = train_user_items.get(user, set())
    return [p for p in recs if p not in bought][:k]

def rrf_fuse(ranked_lists, k_param=RRF_K, top_k=10):
    scores = {}
    for lst in ranked_lists:
        for rank, item in enumerate(lst, start=1):
            scores[item] = scores.get(item, 0.0) + 1.0 / (k_param + rank)
    return [item for item, _ in sorted(scores.items(), key=lambda x: -x[1])[:top_k]]

def hybrid_recommend(user, k=10):
    seg_list     = segment_rules_component(user, k)        # (b)
    cf_list      = cf_recommend(user, k)                    # (c)
    content_list = content_recommend(user, k)               # (d)
    fused = rrf_fuse([seg_list, cf_list, content_list], k_param=RRF_K, top_k=k)
    if len(fused) < k:
        fillers = [p for p in most_popular_recommend(user, k) if p not in fused]
        fused += fillers[:k - len(fused)]
    return fused[:k]

## Section 9 — Run all 6 systems and write the comparison table

In [ ]:
K = 10
recommenders = [
    ('Most-Popular',     most_popular_recommend),
    ('Category-Popular', category_popular_recommend),
    ('Apriori-only',     apriori_recommend),
    ('FP-Growth-only',   fpgrowth_recommend),
    ('CF-only',          cf_recommend),
    ('Hybrid (RRF)',     hybrid_recommend),
]

agg_rows, per_user_all = [], []
for name, fn in recommenders:
    agg, per_user = run_eval(fn, name, test_users, test_user_items, catalog_size, k=K)
    agg_rows.append(agg)
    per_user['system'] = name
    per_user_all.append(per_user)

comparison_df = pd.DataFrame(agg_rows)
comparison_df.to_csv('../outputs/m5/comparison_table.csv', index=False)
print('\nComparison table:')
print(comparison_df.to_string(index=False))

per_user_df = pd.concat(per_user_all, ignore_index=True).rename(
    columns={'precision_at_k': 'precision_at_10', 'recall_at_k': 'recall_at_10'}
)
per_user_df.to_csv('../outputs/m5/per_user_metrics.csv', index=False)

## Section 10 — Significance testing (Wilcoxon + bootstrap CI + Cliff's delta)

In [ ]:
from scipy.stats import wilcoxon

def cliffs_delta(x, y):
    n_gt = int(np.sum(x > y))
    n_lt = int(np.sum(x < y))
    n = len(x)
    return (n_gt - n_lt) / n if n else 0.0

def bootstrap_ci(diff, n_boot=1000, alpha=0.05, seed=42):
    rng = np.random.default_rng(seed)
    n = len(diff)
    means = np.empty(n_boot)
    for i in range(n_boot):
        sample = rng.choice(diff, size=n, replace=True)
        means[i] = sample.mean()
    return float(np.quantile(means, alpha / 2)), float(np.quantile(means, 1 - alpha / 2))

pivot = per_user_df.pivot_table(
    index='customer_unique_id', columns='system', values='precision_at_10', fill_value=0.0
)

HYBRID = 'Hybrid (RRF)'
sig_rows = []
for other in [c for c in pivot.columns if c != HYBRID]:
    x = pivot[HYBRID].to_numpy()
    y = pivot[other].to_numpy()
    diff = x - y
    if np.all(diff == 0):
        w_p = 1.0
    else:
        try:
            _, w_p = wilcoxon(x, y, zero_method='wilcox', alternative='two-sided')
            w_p = float(w_p)
        except ValueError:
            w_p = float('nan')
    ci_low, ci_high = bootstrap_ci(diff, n_boot=1000)
    sig_rows.append({
        'comparison':   f'Hybrid (RRF) vs {other}',
        'wilcoxon_p':   w_p,
        'mean_diff':    float(diff.mean()),
        'ci_low':       ci_low,
        'ci_high':      ci_high,
        'cliffs_delta': cliffs_delta(x, y),
        'n_pairs':      int(len(diff)),
    })

sig_df = pd.DataFrame(sig_rows)
sig_df.to_csv('../outputs/m5/significance_results.csv', index=False)
print('\nSignificance results:')
print(sig_df.to_string(index=False))

# Sample recommendations for the paper
sample_users = list(test_users)[:20]
rec_rows = []
for u in sample_users:
    for name, fn in recommenders:
        recs = fn(u, K)
        rec_rows.append({'customer_unique_id': u, 'system': name, 'top_10': '|'.join(recs)})
pd.DataFrame(rec_rows).to_csv('../outputs/m5/recommendations_sample.csv', index=False)
print('\nWrote: outputs/m5/{comparison_table, per_user_metrics, significance_results, recommendations_sample}.csv')